In [ ]:
from adaptive_latents import StreamingKalmanFilter, StimRegressor, BaseMultiKernelRegressor, ArrayWithTime
from adaptive_latents.stim_designer import *
from adaptive_latents.input_sources.lds_simulation import LDS
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

rng = np.random.default_rng()

In [ ]:
_, observations, stimulations = LDS.run_nest_dynamical_system(rotations=100, u_function='curvy')

In [ ]:
stimulations.t[10]

In [ ]:
t = stimulations.t[10]

In [ ]:
_, observations, stimulations = LDS.run_nest_dynamical_system(rotations=100, u_function='curvy')


def angle_beween(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    angle = np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))
    return angle

new_stimulations = []
for i, stimulation in enumerate(stimulations):
    new_stimulation = np.zeros(3)
    if stimulation.any():
        new_stimulation[2] = stimulation[0]
    elif i > 0 and stimulations[i-1].any():
        a_between = 0
        while a_between < 5 * np.pi/180:
            new_stimulation = rng.normal(size=(3,))
            new_stimulation = new_stimulation / np.linalg.norm(new_stimulation)
            new_stimulation = np.abs(new_stimulation)
            a_between = angle_beween(new_stimulation, np.array([0,0,1]))
    new_stimulations.append(ArrayWithTime(new_stimulation,stimulation.t))
stimulations = ArrayWithTime.from_list(new_stimulations)


sr = StimRegressor(
    autoreg=StreamingKalmanFilter(steps_between_refits=5),
    stim_reg=BaseMultiKernelRegressor(maxlen=20),
)

sr.offline_run_on([(observations, 'X'),  (stimulations, 'stim')])

In [ ]:
sr.stim_reg.output_history[:sr.stim_reg.n_observed]
# sr.stim_reg.input_histories

In [ ]:
f = lambda u: sr.stim_reg.make_jax_pred_f([np.array([1,0,0]), u, observations.t[-1]])

In [ ]:
goal = np.zeros((3,1))
goal[2] = 1

stim_designer_closed_loop = StimDesigner(optimization_method='jaxopt')
regressed_u_to_s_function = sr.stim_reg.make_jax_pred_f()
u, l = design_stim_jaxopt(
    v=goal,
    u_dimension=observations.shape[1],
    u_to_s_function=regressed_u_to_s_function,
    rng=rng,
)


In [ ]:
test_points = []
for _ in range(5000):
    test_point = np.abs(rng.normal(size=(3,1)))
    test_point = test_point / np.linalg.norm(test_point)
    test_points.append(test_point)
test_points = np.squeeze(test_points)

In [ ]:
def objective(u):
    s = u
    # s = regressed_u_to_s_function(s)
    s_norm = jnp.linalg.norm(s)
    loss = 0
    loss += 0.001 * (30 - jnp.sum(jnp.abs(u)))
    loss += jnp.dot(s, goal) / (s_norm + 1e-10)
    return -loss.reshape()

c = [-objective(u) for u in test_points]


%matplotlib qt
fig, axs = plt.subplots(subplot_kw=dict(projection='3d'), squeeze=False)
ax = axs[0,0]

ax.scatter(test_points[:,0], test_points[:,1], test_points[:,2], c=c)
